# Neural Network Activation Functions and Their Derivatives

Each section plots an activation function beside its derivative. Run the notebook from top to bottom.

**Requirements:** `numpy` and `matplotlib`  
If needed, install them with `!pip install numpy matplotlib`.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Shared input range for all plots
x = np.linspace(-6, 6, 1000)

def plot_pair(name, f, df, formula, derivative):
    y = f(x)
    dy = df(x)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle(name, fontsize=16, fontweight="bold")

    axes[0].plot(x, y, linewidth=2)
    axes[0].set_title(f"Activation: {formula}")
    axes[0].set_xlabel("x")
    axes[0].set_ylabel(f"{name}(x)")
    axes[0].grid(True, alpha=0.3)
    axes[0].axhline(0, color="black", linewidth=0.7)
    axes[0].axvline(0, color="black", linewidth=0.7)

    axes[1].plot(x, dy, linewidth=2, color="orange")
    axes[1].set_title(f"Derivative: {derivative}")
    axes[1].set_xlabel("x")
    axes[1].set_ylabel(f"d/dx [{name}(x)]")
    axes[1].grid(True, alpha=0.3)
    axes[1].axhline(0, color="black", linewidth=0.7)
    axes[1].axvline(0, color="black", linewidth=0.7)

    plt.tight_layout()
    plt.show()


## Sigmoid

- **Activation:** `f(x) = 1 / (1 + e^(-x))`
- **Derivative:** `f'(x) = f(x)(1 - f(x))`

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

def sigmoid_derivative(z):
    s = sigmoid(z)
    return s * (1 - s)

plot_pair("Sigmoid", sigmoid, sigmoid_derivative,
          "1 / (1 + exp(-x))", "sigmoid(x) * (1 - sigmoid(x))")

## Tanh

- **Activation:** `f(x) = tanh(x)`
- **Derivative:** `f'(x) = 1 - tanh²(x)`

In [ ]:
def tanh(z):
    return np.tanh(z)

def tanh_derivative(z):
    return 1 - np.tanh(z)**2

plot_pair("Tanh", tanh, tanh_derivative,
          "tanh(x)", "1 - tanh(x)^2")

## ReLU

- **Activation:** `f(x) = max(0, x)`
- **Derivative:** `f'(x) = 0 for x < 0; 1 for x > 0`

In [ ]:
def relu(z):
    return np.maximum(0, z)

def relu_derivative(z):
    # At x = 0, the derivative is conventionally set to 0 in this implementation.
    return (z > 0).astype(float)

plot_pair("ReLU", relu, relu_derivative,
          "max(0, x)", "0 (x <= 0), 1 (x > 0)")

## Leaky ReLU

- **Activation:** `f(x) = x if x > 0 else 0.01x`
- **Derivative:** `f'(x) = 1 if x > 0 else 0.01`

In [ ]:
LEAKY_ALPHA = 0.01

def leaky_relu(z):
    return np.where(z > 0, z, LEAKY_ALPHA * z)

def leaky_relu_derivative(z):
    return np.where(z > 0, 1.0, LEAKY_ALPHA)

plot_pair("Leaky ReLU", leaky_relu, leaky_relu_derivative,
          "x (x > 0), 0.01x (x <= 0)", "1 (x > 0), 0.01 (x <= 0)")

## ELU

- **Activation:** `f(x) = x if x > 0 else alpha(exp(x) - 1)`
- **Derivative:** `f'(x) = 1 if x > 0 else alpha exp(x)`

In [ ]:
ELU_ALPHA = 1.0

def elu(z):
    return np.where(z > 0, z, ELU_ALPHA * np.expm1(z))

def elu_derivative(z):
    return np.where(z > 0, 1.0, ELU_ALPHA * np.exp(z))

plot_pair("ELU", elu, elu_derivative,
          "x (x > 0), alpha(exp(x) - 1) (x <= 0)",
          "1 (x > 0), alpha*exp(x) (x <= 0)")

## SELU

- **Activation:** `f(x) = scale * (x if x > 0 else alpha(exp(x) - 1))`
- **Derivative:** `f'(x) = scale if x > 0 else scale*alpha*exp(x)`

In [ ]:
SELU_ALPHA = 1.6732632423543772
SELU_SCALE = 1.0507009873554805

def selu(z):
    return SELU_SCALE * np.where(
        z > 0, z, SELU_ALPHA * np.expm1(z)
    )

def selu_derivative(z):
    return SELU_SCALE * np.where(
        z > 0, 1.0, SELU_ALPHA * np.exp(z)
    )

plot_pair("SELU", selu, selu_derivative,
          "scale * (x if x > 0 else alpha*(exp(x)-1))",
          "scale (x > 0), scale*alpha*exp(x) (x <= 0)")

## Softplus

- **Activation:** `f(x) = ln(1 + exp(x))`
- **Derivative:** `f'(x) = sigmoid(x)`

In [ ]:
def softplus(z):
    # Numerically stable form of log(1 + exp(z))
    return np.logaddexp(0, z)

def softplus_derivative(z):
    return sigmoid(z)

plot_pair("Softplus", softplus, softplus_derivative,
          "log(1 + exp(x))", "sigmoid(x)")

## Swish / SiLU

- **Activation:** `f(x) = x * sigmoid(x)`
- **Derivative:** `f'(x) = sigmoid(x) + x*sigmoid(x)*(1-sigmoid(x))`

In [ ]:
def swish(z):
    return z * sigmoid(z)

def swish_derivative(z):
    s = sigmoid(z)
    return s + z * s * (1 - s)

plot_pair("Swish / SiLU", swish, swish_derivative,
          "x * sigmoid(x)", "sigmoid(x) + x*sigmoid(x)*(1-sigmoid(x))")

## GELU (tanh approximation)

- **Activation:** `f(x) ≈ 0.5x[1 + tanh(sqrt(2/pi)(x + 0.044715x³))]`
- **Derivative:** `Derivative of the tanh approximation`

In [ ]:
GELU_C = np.sqrt(2 / np.pi)
GELU_K = 0.044715

def gelu(z):
    u = GELU_C * (z + GELU_K * z**3)
    return 0.5 * z * (1 + np.tanh(u))

def gelu_derivative(z):
    u = GELU_C * (z + GELU_K * z**3)
    t = np.tanh(u)
    du = GELU_C * (1 + 3 * GELU_K * z**2)
    return 0.5 * (1 + t) + 0.5 * z * (1 - t**2) * du

plot_pair("GELU (tanh approximation)", gelu, gelu_derivative,
          "0.5*x*(1 + tanh(sqrt(2/pi)*(x + 0.044715*x^3)))",
          "Derivative of the tanh approximation")

## Notes

- The left panel shows the activation function; the right panel shows its derivative.
- For ReLU and Leaky ReLU, the derivative at exactly `x = 0` is not uniquely defined mathematically. The code uses a conventional value for plotting.
- GELU here uses the commonly used tanh approximation, and its derivative is the derivative of that approximation.
- ELU uses `alpha = 1`; Leaky ReLU uses a negative-side slope of `0.01`.
